# Slowly Changing Dimension (SCD) Type 2 with Delta Lake

**Slowly Changing Dimensions (SCD)** track how dimension attributes change over time.  
**Type 2** preserves full history by adding a new row for each change, while keeping the old row with an end date.

### SCD Type 2 Row structure
| Column | Purpose |
|---|---|
| `surrogate_key` | Auto-generated unique row identifier |
| `id` | Natural/business key from source |
| `name` | The tracked attribute |
| `effective_start` | When this version became active |
| `effective_end` | When this version was superseded (`NULL` = current) |
| `is_current` | Boolean flag for the active record |

### Techniques covered
1. Initial load — seed the SCD table from source CSV
2. Simulated change — update a region name in the source
3. SCD Type 2 MERGE — expire old rows, insert new versions
4. Query current records
5. Query full history for a key
6. Time-travel with Delta Lake (`VERSION AS OF`)
7. Audit the Delta transaction log

## Setup — Spark Session

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark

spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)

## 1. Read the source CSV

The source file contains five world regions with their IDs and names.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

CSV_PATH = "/Volumes/population_metrics/landing/datasets/countries_dataset/countries_dataset/csv_data/country_regions/country_regions.csv"

source_schema = StructType([
    StructField("id",   IntegerType(), False),
    StructField("name", StringType(),  True)
])

source_df = spark.read.format("csv").schema(source_schema).option("header", True).load(CSV_PATH)
source_df.show()

## 2. Create the SCD Type 2 Delta table

We add surrogate key, effective date range and a `is_current` flag to the raw columns.

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS population_metrics.default.scd_country_regions (
        surrogate_key  BIGINT GENERATED ALWAYS AS IDENTITY,
        id             INT     NOT NULL,
        name           STRING,
        effective_start TIMESTAMP NOT NULL,
        effective_end   TIMESTAMP,
        is_current      BOOLEAN   NOT NULL
    )
    USING DELTA
    COMMENT 'SCD Type 2 dimension table for world regions'
""")

print("Table created (or already exists).")

## 3. Initial Load — seed the SCD table

On first load every source row becomes a current, open-ended record (`effective_end = NULL`, `is_current = TRUE`).

In [ ]:
from pyspark.sql import functions as F

# Only seed when the table is empty
row_count = spark.table("population_metrics.default.scd_country_regions").count()

if row_count == 0:
    initial_df = source_df.select(
        F.col("id"),
        F.col("name"),
        F.current_timestamp().alias("effective_start"),
        F.lit(None).cast("timestamp").alias("effective_end"),
        F.lit(True).alias("is_current")
    )

    initial_df.write.format("delta").mode("append").saveAsTable("population_metrics.default.scd_country_regions")
    print(f"Seeded {initial_df.count()} rows.")
else:
    print(f"Table already has {row_count} rows — skipping initial load.")

spark.table("population_metrics.default.scd_country_regions").show(truncate=False)

## 4. Simulate a source change

Imagine the business decides to rename **"America"** to **"North & South America"** and add a new region **"Antarctica"** (id = 60).

We build an in-memory DataFrame representing the new state of the source — this is what would arrive from the upstream system.

In [ ]:
from pyspark.sql import Row

# Updated source snapshot (America renamed, Antarctica added)
updated_rows = [
    Row(id=10, name="North & South America"),  # changed
    Row(id=20, name="Europe"),                 # unchanged
    Row(id=30, name="Asia"),                   # unchanged
    Row(id=40, name="Oceania"),                # unchanged
    Row(id=50, name="Africa"),                 # unchanged
    Row(id=60, name="Antarctica"),             # new
]

updated_source_df = spark.createDataFrame(updated_rows, schema=source_schema)
updated_source_df.show(truncate=False)

## 5. SCD Type 2 MERGE

The MERGE performs two actions in a single atomic operation:

- **WHEN MATCHED** and the name has changed → expire the current row (`effective_end = now`, `is_current = FALSE`)
- **WHEN NOT MATCHED** by target → insert a brand-new current row

After expiring a row we need a second pass to insert the new version of the changed record.

In [ ]:
from delta.tables import DeltaTable

scd_table = DeltaTable.forName(spark, "population_metrics.default.scd_country_regions")

# ── Step 1: Expire rows whose name changed ──────────────────────────────────
scd_table.alias("target").merge(
    updated_source_df.alias("source"),
    "target.id = source.id AND target.is_current = TRUE"
).whenMatchedUpdate(
    condition="target.name <> source.name",
    set={
        "effective_end": "current_timestamp()",
        "is_current":    "FALSE"
    }
).whenNotMatchedInsert(
    # Completely new ids (e.g. Antarctica)
    values={
        "id":              "source.id",
        "name":            "source.name",
        "effective_start": "current_timestamp()",
        "effective_end":   "NULL",
        "is_current":      "TRUE"
    }
).execute()

print("Step 1 complete — old rows expired, new ids inserted.")

# ── Step 2: Insert new current version for changed rows ─────────────────────
# Find ids that were just expired (is_current = FALSE but no matching current row)
expired_ids = (
    spark.table("population_metrics.default.scd_country_regions")
    .filter("is_current = FALSE")
    .select("id")
    .join(
        spark.table("population_metrics.default.scd_country_regions").filter("is_current = TRUE").select("id"),
        on="id", how="left_anti"
    )
)

new_versions_df = (
    updated_source_df.alias("src")
    .join(expired_ids.alias("exp"), on="id", how="inner")
    .select(
        F.col("src.id"),
        F.col("src.name"),
        F.current_timestamp().alias("effective_start"),
        F.lit(None).cast("timestamp").alias("effective_end"),
        F.lit(True).alias("is_current")
    )
)

if new_versions_df.count() > 0:
    new_versions_df.write.format("delta").mode("append").saveAsTable("population_metrics.default.scd_country_regions")
    print(f"Step 2 complete — inserted {new_versions_df.count()} new current version(s).")
else:
    print("Step 2 — no changed rows to re-insert.")

## 6. Query current records

Filter `is_current = TRUE` to see the live dimension — exactly what a BI report would join to.

In [ ]:
spark.sql("""
    SELECT surrogate_key, id, name, effective_start, effective_end, is_current
    FROM population_metrics.default.scd_country_regions
    WHERE is_current = TRUE
    ORDER BY id
""").show(truncate=False)

## 7. Query full history for a specific key

Retrieve all versions of region `id = 10` (America → North & South America) to see the audit trail.

In [ ]:
spark.sql("""
    SELECT surrogate_key, id, name, effective_start, effective_end, is_current
    FROM population_metrics.default.scd_country_regions
    WHERE id = 10
    ORDER BY effective_start
""").show(truncate=False)

## 8. Delta Lake Time Travel — point-in-time snapshot

Delta Lake keeps every version of the table.  
Use `VERSION AS OF` to read the table exactly as it looked after the initial load (version 1).

In [ ]:
# Show the table at version 1 (right after the initial seed)
spark.sql("""
    SELECT surrogate_key, id, name, effective_start, effective_end, is_current
    FROM population_metrics.default.scd_country_regions VERSION AS OF 1
    ORDER BY id
""").show(truncate=False)

## 9. Audit the Delta transaction log

`DESCRIBE HISTORY` lists every operation that touched the table — useful for lineage and compliance.

In [ ]:
spark.sql("""
    DESCRIBE HISTORY population_metrics.default.scd_country_regions
""").select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

## 10. (Optional) Cleanup — drop the table

Run this cell only if you want to reset the demo and start fresh.

In [ ]:
# Uncomment to reset
# spark.sql("DROP TABLE IF EXISTS population_metrics.default.scd_country_regions")
# print("Table dropped.")